In [1]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    filename="logs/mgmt_operations.as_245.log",
    encoding="utf-8",
)

LOG: logging.Logger = logging.getLogger(__name__)

In [2]:
from epicsarchiver.mgmt.archiver_mgmt_info import ArchivingStatus
from epicsarchiver.mgmt.archiver_mgmt_operations import Storage
from epicsarchiver import ArchiverAppliance
from concurrent.futures import ThreadPoolExecutor

In [3]:

import random

def delete_multiple_pvs(archivers, pv_list):
    
    def delete_pv(args):
        archiver, pv = args
        pv_status = archiver.get_archiving_status(pv)
        if pv_status != ArchivingStatus.BeingArchived:
            LOG.info(f"PV {pv} is not being archived, skipping")
        LOG.info(f"Deleting PV {pv}, first pausing")
        pause_result = archiver.pause_pv(pv)
        LOG.info(f"Pause result: {pv}:{pause_result}")
        delete_result = archiver.delete_pv(pv)
        LOG.info(f"Delete result: {pv}:{delete_result}")
        pv_status = archiver.get_archiving_status(pv)
        LOG.info(f"PV status: {pv}: {pv_status}")
    
    with ThreadPoolExecutor() as executor:
        executor.map(delete_pv, [(ArchiverAppliance(random.choice(archivers).hostname),pv) for pv in pv_list])

In [4]:
archiver_linac_tns = [ArchiverAppliance(f"archiver-linac-0{i}.tn.esss.lu.se") for i in range(2, 9)]

In [5]:

from pathlib import Path

def get_files(folder, ending):
    return [f for f in Path(folder).iterdir() if f.is_file() and f.name.endswith(ending)] 

def read_to_delete_file(filename: Path):
    result = []
    with open(filename, 'r') as file:
        for line in file.readlines():
            if not line.startswith("#"):
                result.append(line.strip().split()[0])
    return result

def read_delete_files(folder):
    files =  get_files(folder, "Verify_PVs.archiver")
    return {f.name: read_to_delete_file(f) for f in files}

def delete_all(archivers, delete_data):
    for _, file_data in delete_data.items():
        delete_multiple_pvs(archivers, file_data)

In [6]:
delete_mbl_data = read_delete_files("AS-245")

In [7]:
delete_mbl_data

{'mbl_090cdl_cryo_plc_010_Verify_PVs.archiver': ['MBL-090Crm:SC-FSM-122:WU_4K_lvl_dec',
  'MBL-090Crm:SC-FSM-122:WU_300K_lvl_dec',
  'MBL-090Crm:SC-FSM-122:PerT310',
  'MBL-090Crm:SC-FSM-122:PerT505',
  'MBL-090Crm:SC-FSM-122:PerT700',
  'MBL-090Crm:SC-FSM-122:PerT712',
  'MBL-090Crm:SC-FSM-122:PerT800',
  'MBL-090Crm:SC-FSM-122:PerRes9',
  'MBL-090Crm:SC-FSM-122:PerRes10',
  'MBL-090CDL:SC-FSM-300:OM_Undefined',
  'MBL-090CDL:SC-FSM-300:OM_Stopped',
  'MBL-090CDL:SC-FSM-300:OM_Purging',
  'MBL-090CDL:SC-FSM-300:OM_Stand_by_4K',
  'MBL-090CDL:SC-FSM-300:OM_Stand_by_2K',
  'MBL-090CDL:SC-FSM-300:OM_Nominal_2K_RF',
  'MBL-090CDL:SC-FSM-300:OM_Starting',
  'MBL-090CDL:SC-FSM-300:OM_Ready_for_RF',
  'MBL-090CDL:SC-FSM-300:OM_RF_OFF',
  'MBL-090CDL:SC-FSM-300:OM_S_CD_300-4K',
  'MBL-090CDL:SC-FSM-300:OM_S_CD_4-2K',
  'MBL-090CDL:SC-FSM-300:OM_S_WU_4-300K',
  'MBL-090CDL:SC-FSM-300:OM_S_WU_2-4K',
  'MBL-090CDL:SC-FSM-300:Rdy_for_pumpdown',
  'MBL-090CDL:SC-FSM-300:Pumpdown_started',
  'MBL-0

In [8]:
delete_all(archiver_linac_tns, delete_mbl_data)